In [1]:
import os
import cv2

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/LUS_model/data"
target_path = r"/mnt/lustre/lalit47/down/seg/LUSS"

# Source folders
train_images_folder = os.path.join(source_path, "train", "images")
train_masks_folder = os.path.join(source_path, "train", "masks")

test_images_folder = os.path.join(source_path, "test", "images")
test_masks_folder = os.path.join(source_path, "test", "masks")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")
test_masks_out = os.path.join(target_path, "test", "masks")

# Create target directories
for folder in [
    train_images_out,
    train_masks_out,
    val_images_out,
    val_masks_out,
    test_images_out,
    test_masks_out,
]:
    os.makedirs(folder, exist_ok=True)


def process_and_save(src, dst, is_mask=False):
    """
    Read image in grayscale,
    resize to 256x256,
    save as PNG.
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    if is_mask:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_NEAREST)
    else:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)

    cv2.imwrite(dst, im)
    return True


def get_file_stems(folder):
    """
    Get file names without extension from a folder.
    """

    valid_exts = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]

    stems = []

    for file in os.listdir(folder):
        file_lower = file.lower()

        if any(file_lower.endswith(ext) for ext in valid_exts):
            stem = os.path.splitext(file)[0]
            stems.append(stem)

    stems.sort()
    return stems


def find_file_by_stem(folder, stem):
    """
    Find file inside folder using filename without extension.
    """

    valid_exts = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]

    for ext in valid_exts:
        file_path = os.path.join(folder, stem + ext)
        if os.path.exists(file_path):
            return file_path

        file_path_upper = os.path.join(folder, stem + ext.upper())
        if os.path.exists(file_path_upper):
            return file_path_upper

    return None


def process_image_mask_pairs(stems, image_folder, mask_folder, img_output, mask_output):
    """
    Process image-mask pairs and save them as PNG.
    """

    for stem in stems:
        image_src = find_file_by_stem(image_folder, stem)
        mask_src = find_file_by_stem(mask_folder, stem)

        image_dst = os.path.join(img_output, stem + ".png")
        mask_dst = os.path.join(mask_output, stem + ".png")

        if image_src is not None:
            process_and_save(image_src, image_dst, is_mask=False)
        else:
            print(f"Image Missing for: {stem}")

        if mask_src is not None:
            process_and_save(mask_src, mask_dst, is_mask=True)
        else:
            print(f"Mask Missing for: {stem}")


# =========================
# Main processing
# =========================

# Train stays train
train_stems = get_file_stems(train_images_folder)

print(f"Train images: {len(train_stems)}")

process_image_mask_pairs(
    train_stems,
    train_images_folder,
    train_masks_folder,
    train_images_out,
    train_masks_out
)

# Test is copied to both val and test
test_stems = get_file_stems(test_images_folder)

print(f"Test images: {len(test_stems)}")
print(f"Validation images: {len(test_stems)}")

# Val = same as test
process_image_mask_pairs(
    test_stems,
    test_images_folder,
    test_masks_folder,
    val_images_out,
    val_masks_out
)

# Test = original test
process_image_mask_pairs(
    test_stems,
    test_images_folder,
    test_masks_folder,
    test_images_out,
    test_masks_out
)

print("\nDataset preprocessing completed successfully!")

Train images: 464
Test images: 100
Validation images: 100

Dataset preprocessing completed successfully!
